# 시장 유형 분류

* 흐름: OD·STAY 집계 → 비교 기준 → K-Means·아키타입 분석 → PCA 시각화
* 입력: 비공개 결합 특성표
* 공개 결과: 저장 출력과 요약 그림

## OD·STAY 집계

## OD·STAY 시간대·목적·출발지 집계

* 집계 단위: 날짜·시간·연령·목적·출발지
* 특수일 제외: 비교 기준 단계에서 적용

In [ ]:
import zipfile
import tempfile
import shutil
import os

import pandas as pd
import numpy as np

In [ ]:
outer_zip_path = "../data/raw/데이터분석 분야_로우데이터.zip"
out_dir = "../data/analysis"
os.makedirs(out_dir, exist_ok=True)

market_od = pd.read_csv("../data/market_od.csv", encoding='utf-8-sig')
code_set = set(market_od["행정기관코드"].astype(int))
FOCUS_DEST = [1162069500, 2623051000]  # 출발지를 연령·목적·시각과 함께 볼 시장권: 신림, 부전

print(len(market_od), len(code_set))

In [ ]:
OD_INNER_ZIPS = [
    "od_20230901_10.zip",
    "od_20230911_20.zip",
    "od_20230921_30.zip",
    "od_20231001_15.zip",
]
STAY_INNER_ZIPS = [
    "stay_20230901_15.zip",
    "stay_20230916_30.zip",
    "stay_20231001_15.zip",
]

# 연령 코드: 0~5는 10세 단위, 6~8(60·70·80대 이상)은 60+로 묶음
AGE_MAP = {
    0: "0-9",
    1: "10-19",
    2: "20-29",
    3: "30-39",
    4: "40-49",
    5: "50-59",
    6: "60+",
    7: "60+",
    8: "60+"
}

# 목적 코드 (SKT 데이터정의서)
PURPOSE_MAP = {
    0: "귀가",
    1: "업무",
    2: "학업",
    3: "쇼핑여가",
    4: "기타",
    5: "여행",
}

In [ ]:
def iter_inner_csv_members_tempfile(outer_zip_path, inner_zip_name, csv_prefix=None):
    # od_20230901_10.zip 등을 임시파일로 저장한 뒤 내부 CSV member 이름을 yield
    with zipfile.ZipFile(outer_zip_path, "r") as outer:
        with outer.open(inner_zip_name) as src:
            fd, tmp_zip_path = tempfile.mkstemp(suffix=".zip")
            os.close(fd)
            try:
                with open(tmp_zip_path, "wb") as f:
                    shutil.copyfileobj(src, f, length=1024 * 1024)  # 1MB 단위 복사

                with zipfile.ZipFile(tmp_zip_path, "r") as inner:
                    members = [m for m in inner.namelist() if m.endswith(".csv")]
                    if csv_prefix is not None:
                        members = [m for m in members if os.path.basename(m).startswith(csv_prefix)]
                    for m in sorted(members):
                        yield tmp_zip_path, m  # tmp_zip_path는 나중에 inner를 다시 열기 위해 반환
            finally:
                if os.path.exists(tmp_zip_path):
                    os.remove(tmp_zip_path)

## OD 집계

In [ ]:
def extract_od_cubes(outer_zip_path, code_set, inner_zip_names, chunksize=500_000):
    # 시장 행정동에 도착한 OD만 남겨 하루 단위로 6가지 집계를 만듦
    usecols = ["origin_hdong_cd", "dest_hdong_cd", "date", "end_time", "age", "dest_purpose", "modal", "od_duration_avg", "od_cnts"]
    out = {"od_hourly": [], "od_purpose": [], "od_origin": [], "od_shop_origin": [], "od_modal": [], "od_focus": []}

    for inner_zip_name in inner_zip_names:
        for tmp_zip_path, csv_member in iter_inner_csv_members_tempfile(outer_zip_path, inner_zip_name, csv_prefix="od_"):
            day = {k: [] for k in out}
            with zipfile.ZipFile(tmp_zip_path, "r") as inner:
                with inner.open(csv_member) as f:
                    for df in pd.read_csv(f, usecols=usecols, chunksize=chunksize):
                        df = df[df["dest_hdong_cd"].isin(code_set) & df["age"].isin(AGE_MAP.keys())].copy()
                        if df.empty:
                            continue
                        df["hour"] = df["end_time"].str.split(":").str[0].astype(int)
                        df["age_band"] = df["age"].map(AGE_MAP)
                        df["modal"] = df["modal"].fillna(-1).astype(int)

                        # 1) 도착 시각 x 연령
                        day["od_hourly"].append(df.groupby(["dest_hdong_cd", "date", "hour", "age_band"], as_index=False)["od_cnts"].sum())
                        # 2) 도착 목적 x 연령 x 도착 시각
                        day["od_purpose"].append(df.groupby(["dest_hdong_cd", "date", "dest_purpose", "age_band", "hour"], as_index=False)["od_cnts"].sum())
                        # 3) 출발 행정동 (전체 목적)
                        day["od_origin"].append(df.groupby(["dest_hdong_cd", "date", "origin_hdong_cd"], as_index=False)["od_cnts"].sum())
                        # 4) 출발 행정동 (쇼핑여가 목적, 평균 이동시간은 건수 가중)
                        shop = df[df["dest_purpose"] == 3].copy()
                        shop["dur_w"] = shop["od_duration_avg"] * shop["od_cnts"]
                        day["od_shop_origin"].append(shop.groupby(["dest_hdong_cd", "date", "origin_hdong_cd"], as_index=False)[["od_cnts", "dur_w"]].sum())
                        # 5) 도착 연령 x 이동수단
                        day["od_modal"].append(df.groupby(["dest_hdong_cd", "date", "age_band", "modal"], as_index=False)["od_cnts"].sum())
                        # 6) 신림·부전 도착의 출발 행정동 x 연령 x 목적 x 도착 시각
                        focus = df[df["dest_hdong_cd"].isin(FOCUS_DEST)]
                        day["od_focus"].append(focus.groupby(["dest_hdong_cd", "date", "origin_hdong_cd", "hour", "age_band", "dest_purpose"], as_index=False)["od_cnts"].sum())

            for k, parts in day.items():
                keys = [c for c in parts[0].columns if c not in ("od_cnts", "dur_w")]
                vals = [c for c in parts[0].columns if c in ("od_cnts", "dur_w")]
                out[k].append(pd.concat(parts).groupby(keys, as_index=False)[vals].sum())
            print(csv_member)

    return {k: pd.concat(v, ignore_index=True) for k, v in out.items()}

In [ ]:
od = extract_od_cubes(outer_zip_path, code_set, OD_INNER_ZIPS)

In [ ]:
for k, v in od.items():
    print(k, v.shape, v["date"].nunique())

In [ ]:
od["od_hourly"].head()

## STAY 집계

In [ ]:
def extract_stay_cubes(outer_zip_path, code_set, inner_zip_names, chunksize=1_000_000):
    # 시장 행정동의 STAY를 연령별 하루 합, 쇼핑여가 목적의 시각별 합으로 집계
    usecols = ["hdong_cd", "date", "time", "age", "purpose", "stay_cnts"]
    out = {"stay_age": [], "stay_shop_hourly": []}

    for inner_zip_name in inner_zip_names:
        for tmp_zip_path, csv_member in iter_inner_csv_members_tempfile(outer_zip_path, inner_zip_name, csv_prefix="stay_"):
            day = {k: [] for k in out}
            with zipfile.ZipFile(tmp_zip_path, "r") as inner:
                with inner.open(csv_member) as f:
                    for df in pd.read_csv(f, usecols=usecols, chunksize=chunksize):
                        df = df[df["hdong_cd"].isin(code_set) & df["age"].isin(AGE_MAP.keys())].copy()
                        if df.empty:
                            continue
                        df["hour"] = df["time"].str.split(":").str[0].astype(int)
                        df["age_band"] = df["age"].map(AGE_MAP)

                        # 1) 연령별 체류 (전체 목적)
                        day["stay_age"].append(df.groupby(["hdong_cd", "date", "age_band"], as_index=False)["stay_cnts"].sum())
                        # 2) 쇼핑여가 체류의 시각 x 연령
                        shop = df[df["purpose"] == 3]
                        day["stay_shop_hourly"].append(shop.groupby(["hdong_cd", "date", "hour", "age_band"], as_index=False)["stay_cnts"].sum())

            for k, parts in day.items():
                keys = [c for c in parts[0].columns if c != "stay_cnts"]
                out[k].append(pd.concat(parts).groupby(keys, as_index=False)["stay_cnts"].sum())
            print(csv_member)

    return {k: pd.concat(v, ignore_index=True) for k, v in out.items()}

In [ ]:
stay = extract_stay_cubes(outer_zip_path, code_set, STAY_INNER_ZIPS)

In [ ]:
for k, v in stay.items():
    print(k, v.shape, v["date"].nunique())

In [ ]:
stay["stay_shop_hourly"].groupby("hour")["stay_cnts"].sum()

* 쇼핑여가 STAY 관측 시간: 09~22시

## 저장

In [ ]:
for k, v in {**od, **stay}.items():
    v.to_csv(f"{out_dir}/{k}.csv", encoding='utf-8-sig', index=False)

## 비교 기준

## 비교 기준

* 기간: 2023.9.1~10.15 중 일반일 38일, 추석 연휴·한글날 제외
* 전국 기준: OD 값이 있는 시장 830행 평균, 동일 행정동의 여러 시장 각각 반영
* 주민 연령: 2023.10.31 행정동 주민등록인구
* 행정동 중심점: 읍면동 경계의 면적 가중 중심점

In [ ]:
import re

import pandas as pd
import numpy as np
import geopandas as gpd
from pyproj import Transformer

In [ ]:
SPECIAL_DATE = {20230928, 20230929, 20230930, 20231001, 20231002, 20231003, 20231009}

market_od = pd.read_csv("../data/market_od.csv", encoding='utf-8-sig')
od_hourly = pd.read_csv("../data/analysis/od_hourly.csv")

### 일반일

In [ ]:
dates = pd.Series(sorted(od_hourly["date"].unique()))
normal = dates[~dates.isin(SPECIAL_DATE)]
is_weekend = pd.to_datetime(normal.astype(str)).dt.dayofweek >= 5

print("전체", len(dates), "/ 일반일", len(normal), "(평일", (~is_weekend).sum(), "· 주말", is_weekend.sum(), ")")

### 전국 기준 시장 행

In [ ]:
od_daily = (
    od_hourly[~od_hourly["date"].isin(SPECIAL_DATE)]
    .groupby("dest_hdong_cd")["od_cnts"].sum() / len(normal)
)

market_base = market_od[["행정기관코드", "시장명", "시도", "시군구", "위도", "경도", "총시장상인"]].copy()
market_base["od_daily"] = market_base["행정기관코드"].map(od_daily)
market_base["markets_in_hdong"] = market_base.groupby("행정기관코드")["시장명"].transform("size")

print("시장", len(market_base), "/ OD 값이 있는 시장 행", market_base["od_daily"].notna().sum())

In [ ]:
market_base[market_base["od_daily"].isna()]

### 주민 연령 구성

In [ ]:
pop = pd.read_csv("../data/og/지역별(행정동) 성별 연령별 주민등록 인구수_20231031.csv", encoding="cp949")
pop = pop.drop_duplicates("행정기관코드")

def age_of(col):
    # "만20세남자", "만110세이상여자" 같은 성별·연령 컬럼에서 나이를 읽음 (남녀를 모두 더해 사용)
    m = re.fullmatch(r"만(\d+)세(이상)?(남자|여자)", col)
    return int(m.group(1)) if m else None

age_cols = {c: age_of(c) for c in pop.columns if age_of(c) is not None}
total = pop[list(age_cols)].sum(axis=1)

def age_sum(lo, hi):
    return pop[[c for c, a in age_cols.items() if lo <= a <= hi]].sum(axis=1)

pop_age = pd.DataFrame({
    "행정기관코드": pop["행정기관코드"],
    "읍면동명": pop["읍면동명"],
    "인구": total,
    "pop_10_19": age_sum(10, 19),
    "pop_20_39": age_sum(20, 39),
    "pop_40_59": age_sum(40, 59),
    "pop_60": age_sum(60, 200),
})
pop_age["share_20_39"] = pop_age["pop_20_39"] / pop_age["인구"]
pop_age["share_60"] = pop_age["pop_60"] / pop_age["인구"]

pop_age.head()

### 행정동 중심점

In [ ]:
kikmix = pd.read_csv("../data/og/KIKmix_20230701.csv", encoding="utf-8-sig")
kikmix = kikmix[kikmix["읍면동명"].notna() & kikmix["말소일자"].isna()].copy()
kikmix["EMD_CD"] = (kikmix["법정동코드"] // 100).astype("int64")
crosswalk = kikmix[["행정동코드", "EMD_CD"]].drop_duplicates()

emd = gpd.read_file("../data/og/emd.shp", encoding="cp949")
emd = emd[~emd.geometry.is_empty].copy()
emd["EMD_CD"] = emd["EMD_CD"].astype("int64")
emd["area"] = emd.geometry.area
emd["cx"] = emd.geometry.centroid.x
emd["cy"] = emd.geometry.centroid.y

In [ ]:
# 법정동 중심점을 면적으로 가중해 행정동 중심점을 만들고 위경도로 변환 (EPSG:5179 -> 4326)
m = crosswalk.merge(emd[["EMD_CD", "area", "cx", "cy"]], on="EMD_CD", how="inner")
m["wx"] = m["cx"] * m["area"]
m["wy"] = m["cy"] * m["area"]
g = m.groupby("행정동코드")[["area", "wx", "wy"]].sum()

transformer = Transformer.from_crs("EPSG:5179", "EPSG:4326", always_xy=True)
lon, lat = transformer.transform((g["wx"] / g["area"]).values, (g["wy"] / g["area"]).values)

centroids = pd.DataFrame({"행정기관코드": g.index, "hdong_lon": lon, "hdong_lat": lat})
print(len(centroids))
centroids.head()

## 저장

In [ ]:
market_base.to_csv("../data/analysis/market_base.csv", encoding='utf-8-sig', index=False)
pop_age.to_csv("../data/analysis/pop_age.csv", encoding='utf-8-sig', index=False)
centroids.to_csv("../data/analysis/hdong_centroids.csv", encoding='utf-8-sig', index=False)

## K-Means와 아키타입 분석

In [ ]:
import argparse

import numpy as np

import pandas as pd

from sklearn.cluster import KMeans

from sklearn.decomposition import PCA

from sklearn.metrics import silhouette_score

BLOCKS = ["A_OD/STAY", "B_메타", "C_접근성", "D_배후지", "E_Huff"]

BINARY = {"지원여부", "조합유무", "먹거리기반", "비식품품목유무", "배달장보기",
          "배후_후보있음", "Huff_local5_결측"}

def block_scores(frame, variables):
    blocks = []
    for block in BLOCKS:
        names = variables.loc[variables.block == block, "variable"].tolist()
        z = {}
        for name in names:
            x = frame[name].astype(float).to_numpy()
            if name in BINARY:
                p = x.mean()
                z[name] = (x - p) / np.sqrt(max(p * (1 - p), 1e-9))
            elif name.startswith("유형_"):
                p = x.mean()
                z[name] = (x - p) / np.sqrt(max(p, 1e-9))
            else:
                z[name] = (x - x.mean()) / x.std(ddof=0)
        array = pd.DataFrame(z).to_numpy()
        pca = PCA(n_components=min(array.shape[1], array.shape[0] - 1), random_state=0).fit(array)
        k = int(np.searchsorted(np.cumsum(pca.explained_variance_ratio_), .80) + 1)
        if block == "C_접근성":
            k = min(k, 2)
        score = pca.transform(array)[:, :k]
        blocks.append(score / np.sqrt(score.var(axis=0, ddof=0).sum()))
    return np.hstack(blocks)

def simplex_projection(values):
    ordered = -np.sort(-values, axis=1)
    cumulative = ordered.cumsum(axis=1)
    index = np.arange(1, values.shape[1] + 1)
    rho = (ordered - (cumulative - 1) / index > 0).sum(axis=1)
    theta = (cumulative[np.arange(values.shape[0]), rho - 1] - 1) / rho
    return np.maximum(values - theta[:, None], 0)

def fit_archetypes(x, k=3, seed=0, iterations=1200):
    """Cutler–Breiman 아키타입 분석 구현."""
    rng = np.random.default_rng(seed)
    n = x.shape[0]
    xtx = x.T @ x
    points = [int(rng.integers(n))]
    distance = ((x - x[points[0]]) ** 2).sum(axis=1)
    for _ in range(k - 1):
        j = int(np.argmax(distance))
        points.append(j)
        distance = np.minimum(distance, ((x - x[j]) ** 2).sum(axis=1))
    b = np.zeros((k, n))
    b[np.arange(k), points] = 1
    a = np.full((n, k), 1 / k)
    for _ in range(iterations):
        q = b @ x
        step_a = 2 * np.linalg.norm(q @ q.T, 2) + 1e-12
        qq, xq = q @ q.T, x @ q.T
        for _ in range(20):
            a = simplex_projection(a - 2 * (a @ qq - xq) / step_a)
        m, atx = a.T @ a, a.T @ x
        step_b = 2 * np.linalg.norm(m, 2) * np.linalg.norm(xtx, 2) + 1e-12
        for _ in range(20):
            q = b @ x
            b = simplex_projection(b - 2 * ((m @ q - atx) @ x.T) / step_b)
    rss = float(((x - a @ (b @ x)) ** 2).sum())
    return rss, a

def classify(features, definitions, first_stage_only=False):
    definitions = pd.concat([definitions, pd.DataFrame([{
        "block": "E_Huff", "variable": "Huff_local5_결측"
    }])], ignore_index=True)
    x = block_scores(features, definitions)
    raw_label = KMeans(n_clusters=2, n_init=50, random_state=0).fit_predict(x)
    urban = raw_label == np.bincount(raw_label).argmax()
    result = features[["market_uid"]].copy()
    result["유형"] = np.where(urban, "도심형", "외곽형")
    print("1차 분류:", result.유형.value_counts().to_dict(),
          "실루엣:", round(silhouette_score(x, raw_label), 3))
    if first_stage_only:
        return result
    xu = x[urban]
    best = min((fit_archetypes(xu, seed=seed) for seed in range(6)), key=lambda item: item[0])
    named = np.array(["쇼핑 밀집형", "독립 근린형", "주거 귀가형"])
    result.loc[urban, "유형"] = named[best[1].argmax(axis=1)]
    result.loc[urban, "최대_원형가중치"] = best[1].max(axis=1)
    print("4유형:", result.유형.value_counts().to_dict())
    return result

VARIABLES = pd.DataFrame([{'block': 'A_OD/STAY', 'variable': 'OD연령_10대이하', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD연령_20-29', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD연령_30-39', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD연령_40-59', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD연령_60+', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD시간_08-11', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD시간_11-15', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD시간_15-17', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD시간_17-20', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD시간_20-24', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD목적_귀가', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD목적_업무', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD목적_학업', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD목적_쇼핑여가', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD목적_기타', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD목적_여행', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD주말비율', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD교통_car', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD교통_bus', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD교통_subway', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD교통_walk', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'STAY연령_10대이하', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'STAY연령_20-29', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'STAY연령_30-39', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'STAY연령_40-59', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'STAY연령_60+', 'unit': '행정동(시장 공유)', 'processing': '비율, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'OD규모_log', 'unit': '행정동(시장 공유)', 'processing': '로그, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY', 'variable': 'STAY규모_log', 'unit': '행정동(시장 공유)', 'processing': '로그, 표준화', 'source': 'cards'},
 {'block': 'A_OD/STAY',
  'variable': '쇼핑STAY연령_0-19',
  'unit': '행정동(시장 공유)',
  'processing': '비율(38일), 표준화',
  'source': 'rawpass stay_shop_hourly'},
 {'block': 'A_OD/STAY',
  'variable': '쇼핑STAY연령_20-29',
  'unit': '행정동(시장 공유)',
  'processing': '비율(38일), 표준화',
  'source': 'rawpass stay_shop_hourly'},
 {'block': 'A_OD/STAY',
  'variable': '쇼핑STAY연령_30-39',
  'unit': '행정동(시장 공유)',
  'processing': '비율(38일), 표준화',
  'source': 'rawpass stay_shop_hourly'},
 {'block': 'A_OD/STAY',
  'variable': '쇼핑STAY연령_40-59',
  'unit': '행정동(시장 공유)',
  'processing': '비율(38일), 표준화',
  'source': 'rawpass stay_shop_hourly'},
 {'block': 'A_OD/STAY',
  'variable': '쇼핑STAY연령_60+',
  'unit': '행정동(시장 공유)',
  'processing': '비율(38일), 표준화',
  'source': 'rawpass stay_shop_hourly'},
 {'block': 'A_OD/STAY',
  'variable': '쇼핑STAY규모_log',
  'unit': '행정동(시장 공유)',
  'processing': '로그, 표준화',
  'source': 'rawpass stay_shop_hourly'},
 {'block': 'B_메타', 'variable': '전체점포_log', 'unit': '시장', 'processing': '로그, 표준화', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '총시장상인_log', 'unit': '시장', 'processing': '로그, 표준화', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '시장면적_log', 'unit': '시장', 'processing': '로그, 표준화', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '노점수_log1p', 'unit': '시장', 'processing': '로그1p, 표준화', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '편의시설수_log1p', 'unit': '시장', 'processing': '로그1p, 표준화', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '점포대상인비율', 'unit': '시장', 'processing': '표준화', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '빈점포율', 'unit': '시장', 'processing': '표준화', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '인접_parking_log1p', 'unit': '시장(주변 시설 수)', 'processing': '로그1p, 표준화', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '인접_bus_log1p', 'unit': '시장(주변 시설 수)', 'processing': '로그1p, 표준화', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '인접_mart_log1p', 'unit': '시장(주변 시설 수)', 'processing': '로그1p, 표준화', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '인접_tour_log1p', 'unit': '시장(주변 시설 수)', 'processing': '로그1p, 표준화', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '인접_conv_log1p', 'unit': '시장(주변 시설 수)', 'processing': '로그1p, 표준화', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '인접_subway_log1p', 'unit': '시장(주변 시설 수)', 'processing': '로그1p, 표준화', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '지원여부', 'unit': '시장', 'processing': '이진, 중심화/√(p(1-p))', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '조합유무', 'unit': '시장', 'processing': '이진, 중심화/√(p(1-p))', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '공동구매참여점포_log1p', 'unit': '시장', 'processing': '로그1p, 표준화', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '품목다양성', 'unit': '시장', 'processing': '표준화', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '먹거리기반', 'unit': '시장', 'processing': '이진, 중심화/√(p(1-p))', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '비식품품목유무', 'unit': '시장', 'processing': '이진, 중심화/√(p(1-p))', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '배달장보기', 'unit': '시장', 'processing': '이진, 중심화/√(p(1-p))', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '같은행정동시장수_log1p', 'unit': '행정동', 'processing': '로그1p, 표준화', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '주민연령비중_청소년', 'unit': '행정동', 'processing': '비율, 표준화', 'source': 'cards(pop_* 집계)'},
 {'block': 'B_메타', 'variable': '주민연령비중_청년', 'unit': '행정동', 'processing': '비율, 표준화', 'source': 'cards(pop_* 집계)'},
 {'block': 'B_메타', 'variable': '주민연령비중_중장년', 'unit': '행정동', 'processing': '비율, 표준화', 'source': 'cards(pop_* 집계)'},
 {'block': 'B_메타', 'variable': '주민연령비중_고령', 'unit': '행정동', 'processing': '비율, 표준화', 'source': 'cards(pop_* 집계)'},
 {'block': 'B_메타', 'variable': '주민수_log', 'unit': '행정동', 'processing': '로그, 표준화', 'source': 'cards(pop_* 합)'},
 {'block': 'B_메타', 'variable': '유형_먹거리/외식형', 'unit': '시장', 'processing': '범주(원-핫), 중심화/√p', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '유형_복합형(생활+비식품)', 'unit': '시장', 'processing': '범주(원-핫), 중심화/√p', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '유형_비식품/전문형', 'unit': '시장', 'processing': '범주(원-핫), 중심화/√p', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '유형_생활형(장보기형)', 'unit': '시장', 'processing': '범주(원-핫), 중심화/√p', 'source': 'cards'},
 {'block': 'B_메타', 'variable': '유형_서비스형', 'unit': '시장', 'processing': '범주(원-핫), 중심화/√p', 'source': 'cards'},
 {'block': 'C_접근성', 'variable': 'AI_min', 'unit': '행정동(AI)', 'processing': '표준화', 'source': 'cards AI_min'},
 {'block': 'C_접근성',
  'variable': 'AI_시도내상대순위',
  'unit': '행정동(시도 내 순위)',
  'processing': '0-1 순위, 표준화',
  'source': 'cards AI_min 파생'},
 {'block': 'E_Huff',
  'variable': 'Huff_기대점유_local5_log',
  'unit': '시장(5km 출발동 가중)',
  'processing': '로그(결측=중앙값 대입), 표준화',
  'source': 'huff_market_features'},
 {'block': 'E_Huff',
  'variable': 'Huff_기대점유_all_log',
  'unit': '시장',
  'processing': '로그, 표준화',
  'source': 'huff_market_features'},
 {'block': 'E_Huff',
  'variable': 'Huff_경쟁강도_log',
  'unit': '시장',
  'processing': '로그, 표준화',
  'source': 'huff_market_features'},
 {'block': 'E_Huff',
  'variable': 'Huff_3km이내시장수_log1p',
  'unit': '시장',
  'processing': '로그1p, 표준화',
  'source': 'huff_market_features'},
 {'block': 'E_Huff', 'variable': '경쟁쌍수_log1p', 'unit': '시장', 'processing': '로그1p, 표준화', 'source': 'cards comp_pairs'},
 {'block': 'E_Huff',
  'variable': '경쟁최근접_km_log',
  'unit': '시장(km)',
  'processing': '로그(결측=p99 대입), 표준화',
  'source': 'cards comp_min_km (결측 336곳)'},
 {'block': 'D_배후지',
  'variable': '배후_후보수_log1p',
  'unit': '시장(행정동 후보 dedup, 없으면 0)',
  'processing': '로그1p, 표준화',
  'source': 'hinterland_search 로직 재현'},
 {'block': 'D_배후지',
  'variable': '배후_인구_log1p',
  'unit': '시장(후보 인구 합, 없으면 0)',
  'processing': '로그1p, 표준화',
  'source': 'hinterland_search 로직 재현'},
 {'block': 'D_배후지',
  'variable': '배후_후보있음',
  'unit': '시장(이진)',
  'processing': '이진, 중심화/√(p(1-p))',
  'source': 'hinterland_search 로직 재현'},
 {'block': 'D_배후지',
  'variable': '배후_코사인평균',
  'unit': '시장(후보 없으면 후보 있는 시장 중앙값)',
  'processing': '표준화',
  'source': '결측 대입 아님, 중앙값 중립값'},
 {'block': 'D_배후지',
  'variable': '배후_코사인최대',
  'unit': '시장(후보 없으면 후보 있는 시장 중앙값)',
  'processing': '표준화',
  'source': '결측 대입 아님, 중앙값 중립값'},
 {'block': 'D_배후지',
  'variable': '배후_최근접거리',
  'unit': '시장(km, 후보 없으면 후보 있는 시장 중앙값)',
  'processing': '표준화',
  'source': '결측 대입 아님, 중앙값 중립값'}])

In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display, Image
from plotting import FIGS, plot_type_counts, plot_pca

features_file = os.environ.get("BC_MARKET_FEATURES")
if features_file and Path(features_file).exists():
    features = pd.read_csv(features_file, encoding="utf-8-sig")
    variables = VARIABLES
    classified = classify(features, variables)
    counts = classified["유형"].value_counts().to_dict()
    display(pd.DataFrame({"유형": list(counts), "시장 수": list(counts.values())}))
else:
    print("결합 특성표가 공개 폴더에 없어 유형 분류는 재실행하지 않았습니다.")
    saved_counts = pd.DataFrame({"유형": ["외곽형", "독립 근린형", "주거 귀가형", "쇼핑 밀집형"],
                                 "시장 수": [204, 255, 266, 105]})
    counts = dict(zip(saved_counts["유형"], saved_counts["시장 수"]))
    display(saved_counts)

## 유형 분포와 PCA 투영

* 유형 수·그림: 비공개 입력이 없으면 저장 결과 표시
* PCA 2·3차원: 시각화 용도
* 유형 배정: 결합 주성분 30개 기준

In [ ]:
plot_type_counts(counts)
display(Image(filename=str(FIGS / "fig_v4_0_type_counts.png")))

scores_file = os.environ.get("BC_COMPONENT_SCORES")
labels_file = os.environ.get("BC_CLASSIFIED_MARKETS")
if all(x and Path(x).exists() for x in [scores_file, labels_file, features_file]):
    plot_pca(Path(scores_file), Path(labels_file), Path(features_file),
             FIGS / "fig_v4_0_pca_2d_3d.png")
else:
    print("비공개 결합 점수와 시장 특성표가 없어 PCA 그림은 저장된 결과를 표시합니다.")
display(Image(filename=str(FIGS / "fig_v4_0_pca_2d_3d.png")))